## Prep

In [11]:
import json
import sys
import pandas as pd
import collections 
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import numpy as np
from itertools import chain
from itertools import combinations
sys.path.insert(0, '..')
from src.experiment_utils.helper_classes import token, span, repository
from src.d02_corpus_statistics.corpus import Corpus
import types
from owlready2 import sync_reasoner
from collections import Counter
from transformers import pipeline, AutoModel, PreTrainedTokenizerBase
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
from transformers.modeling_outputs import TokenClassifierOutput
import spacy
from spacy.training import offsets_to_biluo_tags
from datasets import Dataset, DatasetDict
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import evaluate
from typing import Any, Dict, List
from sklearn.metrics import f1_score
import math

cwd = os.getcwd()
pol_dir = cwd+"/../src/d01_data"

so many things to try even just for NER, especially given how long some of these token sequences are

trying multiple classifiction heads
- different base models
- using last hidden state vs using last few hidden states (average or concatenation)
- weighted vs unweighted loss
- evaluation metrics (token micro F1 or overlap instead of seqeval)

#### Create Dataset

In [2]:
pol_df = pd.read_pickle(pol_dir+"/preprocessed_dataframe.pkl")[["Policy","Text","Tokens","Curation"]]

https://medium.com/@shahrukhx01/multi-task-learning-with-transformers-part-1-multi-prediction-heads-b7001cf014bf

In [3]:
def span_to_bio_tok_lbls(feature_name, tokens, spans, label2id):
    token_labels = ["O"] * len(tokens)
    for spn in spans:
        if spn.feature == feature_name:
            start_char = spn.start
            end_char = spn.stop
            inside_tokens = []
            for i, tok in enumerate(tokens):
                tok_start = tok.start
                tok_end = tok.stop
                overlap = not (tok_end <= start_char or tok_start >= end_char)
                if overlap:
                    inside_tokens.append(i)
            if inside_tokens:
                token_labels[inside_tokens[0]] = f"B"
                for i in inside_tokens[1:]:
                    token_labels[i] = f"I"
    return [label2id[l] for l in token_labels]

def df_to_dataset(df):
    label2id = {
        "O":0, "B":1, "I":2
    }
    dataset = {
        "id":[],
        "text":[],
        "tokens":[],
        "labels_Actor":[],
        "labels_InstrumentType":[],
        "labels_Objective":[],
        "labels_Resource":[],
        "labels_Time":[]
    }
    for artid in df.index:
        tokens = df.loc[artid,"Tokens"]
        if len(tokens) <= 512: # we'll change this eventually
            text = df.loc[artid,"Text"]
            spans = df.loc[artid,"Curation"]
            token_texts = [t.text for t in tokens]
            dataset['id'].append(artid)
            dataset["text"].append(text)
            dataset["tokens"].append(token_texts)
            for ftr in ["Actor", "InstrumentType", "Objective", "Resource", "Time"]:
                token_level_labels = span_to_bio_tok_lbls(ftr, tokens, spans, label2id)
                dataset[f"labels_{ftr}"].append(token_level_labels)
    return Dataset.from_dict(dataset), list(label2id)

In [4]:
dataset, label_list = df_to_dataset(pol_df)
id2label = {}
label2id = {}
for i, lbl in enumerate(label_list):
    id2label[i] = lbl
    label2id[lbl] = i
# do the datasets need to differ by model used for tokenization of results too??

In [ ]:
'''
for r in [0,1,2]:
    td_test = dataset.train_test_split(test_size=0.2, seed=r)
    train_dev = td_test['train'].train_test_split(test_size=0.25, seed=r)
    ds_dct = DatasetDict({"train":train_dev['train'], "dev":train_dev['test'], "test":td_test['test']})
    print(ds_dct)
    ds_dct.save_to_disk(cwd+f"/inputs/sep/dsdct_r{r}")
'''

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 10187.69 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 12580.75 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 12093.87 examples/s]


''

#### Tokenize

In [12]:
label2id = {
        "O":0, "B":1, "I":2
    }
id2label = {
    0:"O", 1:"B", 2:"I"
}

In [13]:
model_name = "microsoft/deberta-v3-base" # suggested lr of 3e-5
#model_name = "dslim/bert-base-NER-uncased"
#model_name = "FacebookAI/xlm-roberta-base"

have to adapt the tokenizing and aligning script to account for the separate label lists

In [14]:
# all feature/label types
label_cols = [
    "labels_Actor",
    "labels_InstrumentType",
    "labels_Objective",
    "labels_Resource",
    "labels_Time"
]

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    # adapted for multi-head from https://huggingface.co/docs/transformers/en/tasks/token_classification
    # even tho the token lists area already split into words, we need to break them into subwords
    # and then ensure that the label sequences still align in the new token sequence
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, padding=True, return_attention_mask=True)
    # for each label type/list
    for col in label_cols:
        all_aligned_labels = []
        # loop through this label type's sequence in each sample and realign
        for sample_idx, labels in enumerate(examples[col]):
            word_ids = tokenized_inputs.word_ids(batch_index=sample_idx)
            # smth like [None, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 19, 20]
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(labels[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            all_aligned_labels.append(label_ids)
        tokenized_inputs[col] = all_aligned_labels
    return tokenized_inputs

In [15]:
r=0
dataset_dict = DatasetDict.load_from_disk(cwd+f"/inputs/sep/dsdct_r{r}")
tokenized_dsdct = dataset_dict.map(tokenize_and_align_labels, batched=True)
#tokenized_dsdct.set_format(type="torch", columns=["input_ids", "attention_mask"] + label_cols)

In [16]:
# sanity checkign
tokenized_inputs = tokenizer(dataset_dict['train'][0:5]["tokens"], truncation=True, is_split_into_words=True)
for sample_idx, labels in enumerate(dataset_dict['train'][0:5]['labels_Actor']):
    word_ids = tokenized_inputs.word_ids(batch_index=sample_idx)
    print(word_ids)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[None, 0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 29, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, None]
[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 

new custom data collator for multi-heads

we need a new data collator because we have mutliple label lists

In [17]:
class MultiHeadDataCollator:
    '''
    Using PreTrainedTokenizerBase i.e. whatever pretrained tokenizer we have from tokenize_and_align_labels
    And using pad_sequence
    '''
    def __init__(self, tokenizer: PreTrainedTokenizerBase, label_columns: List[str], padding=True, max_length=None):
        #initializing the essentials
        self.tokenizer = tokenizer
        self.label_columns = label_columns
        self.padding = padding
        self.max_length = max_length
    def __call__(self, features):
        input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        #attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features] # something isnt working
        attention_mask = [
            torch.tensor(f.get("attention_mask", [1]*len(f["input_ids"])), dtype=torch.long)
            for f in features
        ]
        # padding inputids and attnmask
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        # batch
        batch = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }
        # padding labels
        for col in self.label_columns:
            #label_lists = [torch.tensor(f[col], dtype=torch.long) for f in features] # something isnt working here either
            label_lists = [
                torch.tensor(f.get(col, [-100]*len(f["input_ids"])), dtype=torch.long)
                for f in features
            ]
            labels_padded = pad_sequence(label_lists, batch_first=True, padding_value=-100)
            # then finally adding to batch
            batch[col] = labels_padded
        return batch

In [18]:
# sanity checking
data_collator = MultiHeadDataCollator(tokenizer=tokenizer, label_columns=label_cols, max_length=512)
sample_batch = [tokenized_dsdct['train'][i] for i in range(16)]
batch = data_collator(sample_batch)
for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([16, 507])
attention_mask torch.Size([16, 507])
labels_Actor torch.Size([16, 507])
labels_InstrumentType torch.Size([16, 507])
labels_Objective torch.Size([16, 507])
labels_Resource torch.Size([16, 507])
labels_Time torch.Size([16, 507])


In [19]:
#sanity checking
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
for idx, tok in enumerate(tokens[:50]):
    print(f"{idx:03} | {tok:15} | "
          f"A:{batch['labels_Actor'][0][idx].item():2}  "
          f"T:{batch['labels_Time'][0][idx].item():2}  "
          f"I:{batch['labels_InstrumentType'][0][idx].item():2}")


000 | [CLS]           | A:-100  T:-100  I:-100
001 | ▁article        | A: 0  T: 0  I: 0
002 | ▁18             | A: 0  T: 0  I: 0
003 | ▁bills          | A: 0  T: 0  I: 0
004 | ▁and            | A: 0  T: 0  I: 0
005 | ▁billing        | A: 0  T: 0  I: 0
006 | ▁information    | A: 0  T: 0  I: 0
007 | ▁1              | A: 0  T: 0  I: 0
008 | ▁.              | A: 0  T: 0  I: 0
009 | ▁member         | A: 1  T: 0  I: 0
010 | ▁states         | A: 2  T: 0  I: 0
011 | ▁shall          | A: 0  T: 0  I: 0
012 | ▁ensure         | A: 0  T: 0  I: 0
013 | ▁that           | A: 0  T: 0  I: 0
014 | ▁bills          | A: 0  T: 0  I: 0
015 | ▁and            | A: 0  T: 0  I: 0
016 | ▁billing        | A: 0  T: 0  I: 0
017 | ▁information    | A: 0  T: 0  I: 0
018 | ▁are            | A: 0  T: 0  I: 0
019 | ▁accurate       | A: 0  T: 0  I: 0
020 | ▁,              | A: 0  T: 0  I: 0
021 | ▁easy           | A: 0  T: 0  I: 0
022 | ▁to             | A: 0  T: 0  I: 0
023 | ▁understand     | A: 0  T: 0  I: 0
024 | ▁,  

new model structure

microsoft/deberta-v3-base

#### Weight calculations (currently only using train split)

In [20]:
def get_weights(label_cols, tokenized_dsdct):
    # lets create class (BIO) weights for each feature type
    num_classes = 3
    class_weights = {}
    class_weights_norm = {}
    for lname in label_cols:
        name = lname.replace("labels_", "")
        labels = np.concatenate([np.array(l) for l in tokenized_dsdct['train'][lname]])
        labels = labels[labels != -100]  # remove padding
        counter = Counter(labels)
        # inverse frequency weighting
        weights=[0]*num_classes
        for i in range(num_classes):
            count = counter.get(i, 0)
            if count == 0: #handle no-shows so no zer-os (no dividing by zeros that is)
                weights[i] = 1.0
            else:
                weights[i] = len(labels) / (num_classes * count)
        class_weights[name] = torch.tensor(weights, dtype=torch.float)
        # old vers
        #total = sum(counter.values())
        #class_weights[name] = torch.tensor([total/(num_classes*counter[i]) for i in range(num_classes)], dtype=torch.float)
        # then normalize
        w = torch.tensor(weights, dtype=torch.float)
        w = w / w.mean()
        class_weights_norm[name] = w
    #cls_wt_tot = sum([class_weights[cls] for cls in list(class_weights)])
    #class_weights_norm = {cls: torch.tensor(class_weights[cls]/cls_wt_tot, dtype=torch.float) for cls in list(class_weights)}
    # weights for each head
    head_counts = {}
    for lname in label_cols:
        name = lname.replace("labels_", "")
        # concatenate all labels and remove -100s
        labels = np.concatenate([np.array(l) for l in tokenized_dsdct['train'][lname]])
        labels = labels[labels != -100]
        labels = labels[labels != 0]
        head_counts[name] = len(labels) # only tokens B or I
    total_tokens = sum(head_counts.values())
    head_weights = {head: total_tokens / (len(head_counts) * count) for head, count in head_counts.items()}
    #hd_wt_tot = sum([head_weights[cls] for cls in list(head_weights)])
    #head_weights_norm = {head: math.log(head_weights[head]) for head in list(head_weights)}
    w = torch.tensor(list(head_weights.values()), dtype=torch.float)
    w = w / w.mean()
    head_weights_norm = {head: w[i] for i, head in enumerate(head_weights.keys())}
    return {"class_weights": class_weights, "class_weights_norm": class_weights_norm, "head_weights": head_weights, "head_weights_norm": head_weights_norm}

WEIGHTS = get_weights(label_cols, tokenized_dsdct)
WEIGHTS

{'class_weights': {'Actor': tensor([ 0.3560,  9.8114, 11.2143]),
  'InstrumentType': tensor([ 0.3483, 16.4053, 14.7887]),
  'Objective': tensor([ 0.3513, 53.5521,  7.4079]),
  'Resource': tensor([ 0.3367, 92.7964, 53.0844]),
  'Time': tensor([ 0.3412, 59.8834, 19.0240])},
 'class_weights_norm': {'Actor': tensor([0.0500, 1.3766, 1.5734]),
  'InstrumentType': tensor([0.0331, 1.5603, 1.4066]),
  'Objective': tensor([0.0172, 2.6203, 0.3625]),
  'Resource': tensor([0.0069, 1.9039, 1.0892]),
  'Time': tensor([0.0129, 2.2669, 0.7202])},
 'head_weights': {'Actor': 0.5988807576409815,
  'InstrumentType': 0.8900831733845169,
  'Objective': 0.7447537473233404,
  'Resource': 3.8644444444444446,
  'Time': 1.6522565320665084},
 'head_weights_norm': {'Actor': tensor(0.3864),
  'InstrumentType': tensor(0.5742),
  'Objective': tensor(0.4805),
  'Resource': tensor(2.4931),
  'Time': tensor(1.0659)}}

##### spoiler
weights = {'class_weights': {'Actor': torch.tensor([ 0.3560,  9.8114, 11.2143], dtype=torch.float),
  'InstrumentType': torch.tensor([ 0.3483, 16.4053, 14.7887], dtype=torch.float),
  'Objective': torch.tensor([ 0.3513, 53.5521,  7.4079], dtype=torch.float),
  'Resource': torch.tensor([ 0.3367, 92.7964, 53.0844], dtype=torch.float),
  'Time': torch.tensor([ 0.3412, 59.8834, 19.0240], dtype=torch.float)},
 'class_weights_norm': {'Actor': torch.tensor([-1.0328,  2.2835,  2.4172], dtype=torch.float),
  'InstrumentType': torch.tensor([-1.0548,  2.7976,  2.6939], dtype=torch.float),
  'Objective': torch.tensor([-1.0460,  3.9807,  2.0025], dtype=torch.float),
  'Resource': torch.tensor([-1.0887,  4.5304,  3.9719], dtype=torch.float),
  'Time': torch.tensor([-1.0753,  4.0924,  2.9457], dtype=torch.float)},
 'head_weights': {'Actor': 0.5988807576409815,
  'InstrumentType': 0.8900831733845169,
  'Objective': 0.7447537473233404,
  'Resource': 3.8644444444444446,
  'Time': 1.6522565320665084},
 'head_weights_norm': {'Actor': -0.5126927697303358,
  'InstrumentType': -0.11644036738140337,
  'Objective': -0.2947016557487147,
  'Resource': 1.3518179315899177,
  'Time': 0.5021419487977465}}

since loss is computed per token instead of per span, we'll look at the All instead of the Ents results

## Training components

In [21]:
class DebertaForMultiHeadTokClass(nn.Module):
    def __init__(self):
        super().__init__()
        n_labels = 3
        # shared encoder
        self.base_model = AutoModel.from_pretrained('microsoft/deberta-v3-base')
        hidden_size = self.base_model.config.hidden_size
        #sep linear head for each feature type classification
        self.classifiers = nn.ModuleDict({
            "Actor": nn.Linear(hidden_size, n_labels),
            "InstrumentType": nn.Linear(hidden_size, n_labels),
            "Objective": nn.Linear(hidden_size, n_labels),
            "Resource": nn.Linear(hidden_size, n_labels),
            "Time": nn.Linear(hidden_size, n_labels)
        })
    def forward(self, input_ids, attention_mask=None, **labels):
        # batch of inputs encoded by base model
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        # only uses last hidden state... for now
        # will look into averaging/concatenating last few hidden states
        sequence_output = outputs.last_hidden_state
        # passes encoded input sequence to each classifier to get logits
        logits = {name: self.classifiers[name](sequence_output) for name in self.classifiers}
        loss = None
        if labels:
            loss = 0
            # for labels_Feature, tensor(batch_sz,seq_ln)
            for lname, label in labels.items():
                if label is not None:
                    name = lname.replace("labels_", "")
                    # flatten attn mask
                    active_loss = attention_mask.view(-1) == 1
                    # get active logits, flatten to (num_act_tokens, num_classes) 
                    # then apply active loss mask (to both logits and labels)
                    active_logits = logits[name].view(-1, 3)[active_loss]
                    active_labels = label.view(-1)[active_loss]
                    # weighting BIO classes for this feature
                    #weight = WEIGHTS['class_weights'][name].to(active_logits.device)
                    weight = WEIGHTS['class_weights_norm'][name].to(active_logits.device)
                    loss_fct = nn.CrossEntropyLoss(weight=weight)
                    # computing loss for this head
                    head_loss = loss_fct(active_logits, active_labels)
                    # weight the loss for this head
                    head_loss *= WEIGHTS['head_weights_norm'][name]
                    # sum loss across heads for single update to train simultaneously
                    loss += head_loss
        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
        )

model = DebertaForMultiHeadTokClass()

In [22]:
# work on model that concatenates or averages last few hidden states for encoded representation

new compute_metrics for token micro-f1 instead of seqeval

In [23]:
def compute_metrics_multihead(p):
    prediction_dct, label_dct = p
    lblnames = [i[0] for i in prediction_dct.items()]
    metrics = {}
    # for each head
    for head_name, logits in prediction_dct.items():
        labels = label_dct[lblnames.index(head_name)] # size (batch, seq_len)
        labels_flat = labels.flatten()
        preds_flat = np.argmax(logits, axis=-1).flatten()
        # mask out -100s
        mask = labels_flat != -100
        labels_flat = labels_flat[mask]
        preds_flat = preds_flat[mask]
        # micro F1
        f1 = f1_score(labels_flat, preds_flat, average='micro')
        metrics[f"{head_name}_f1"] = f1
    return metrics

In [24]:
data_collator = MultiHeadDataCollator(tokenizer=tokenizer, label_columns=label_cols, max_length=512)

In [25]:
class MultiHeadTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract labels for each head from the inputs
        labels = {k: inputs.pop(k) for k in list(inputs.keys()) if k.startswith("labels_")}
        # Forward pass
        outputs = model(**inputs, **labels)
        # Your model returns TokenClassifierOutput
        loss = outputs.loss
        if return_outputs:
            return loss, outputs
        return loss

## Where thamagic happpens

In [16]:
training_args = TrainingArguments(
    output_dir=model_name.split("/")[-1],
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_names=label_cols
)

#trainer = Trainer(
trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dsdct["train"],
    eval_dataset=tokenized_dsdct["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_multihead
)
trainer.train()
trainer.save_model(cwd+f"/models/sep/{model_name.split('/')[-1]}_{r}")
del model
del trainer

Epoch,Training Loss,Validation Loss,Actor F1,Instrumenttype F1,Objective F1,Resource F1,Time F1
1,No log,4.805996,0.807211,0.435844,0.594588,0.959933,0.759194
2,No log,3.914751,0.708768,0.481131,0.685723,0.873454,0.801028
3,No log,3.125523,0.718404,0.607194,0.670226,0.832905,0.915609
4,No log,2.822501,0.763289,0.680585,0.652802,0.915208,0.915770
5,No log,2.592992,0.837080,0.714389,0.701220,0.931347,0.964750
6,No log,3.121983,0.842541,0.747952,0.757588,0.978401,0.965232
7,No log,2.366446,0.867272,0.727557,0.704191,0.951903,0.966677
8,No log,2.663637,0.851453,0.737835,0.747551,0.971254,0.966436
9,No log,2.653700,0.876827,0.747390,0.731813,0.971495,0.969167
10,No log,2.842788,0.875542,0.752369,0.737353,0.975751,0.969006


add early stopping to trainer?

In [65]:
label_cols

['labels_Actor',
 'labels_InstrumentType',
 'labels_Objective',
 'labels_Resource',
 'labels_Time']

## Test

old metrics (seqeval)

In [ ]:
from transformers.configuration_utils import PretrainedConfig

class MultiHeadConfig(PretrainedConfig):
    model_type = "deberta-multihead"
    def __init__(self, num_labels=3, heads=None, base_model_name="microsoft/deberta-v3-base", **kwargs):
        super().__init__(**kwargs)
        self.num_labels = num_labels
        self.heads = heads or ["Actor", "InstrumentType", "Objective", "Resource", "Time"]
        self.base_model_name = base_model_name

In [ ]:
mode = "sep"
seqeval = evaluate.load("seqeval")
results_dict = {
    "microsoft/deberta-v3-base":{},
    "FacebookAI/xlm-roberta-base":{},
    "dslim/bert-base-NER-uncased":{}
}
for model_name in list(results_dict):
    results_dict[model_name]["Overall"] = {"precision":[], "recall":[], "f1":[], "accuracy":[]}
    for ftr in ["Actor", "InstrumentType", "Objective", "Resource", "Time"]:
        results_dict[model_name][ftr] = {"precision":[], "recall":[], "f1":[], "number":[]}
for model_name in results_dict:
    for r in [0,1,2]:
        dataset_dict = DatasetDict.load_from_disk(cwd+f"/inputs/{mode}/dsdct_r{r}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model_tt = DebertaForMultiHeadTokClass(model_name)  # your multi-head model
        model_tt.load_state_dict(torch.load(cwd+f"/models/{mode}/{model_name.split('/')[-1]}_{r}/pytorch_model.bin"))
        model_tt.to("cuda")
        model_tt.eval()

        texts = list(dataset_dict['test']['text'])
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to("cuda")

        # get logits per head
        with torch.no_grad():
            outputs = model_tt(**inputs)
            logits = outputs.logits  # dict of tensors per head

        # for each head, compute seqeval metrics
        for head_name, logit_tensor in logits.items():
            # argmax along class dimension
            preds = torch.argmax(logit_tensor, dim=-1).cpu().numpy()

            # get true labels
            labels = np.stack([dataset_dict['test'][f"labels_{head_name}"][i] for i in range(len(dataset_dict['test']))])

            true_predictions = []
            true_labels = []

            for pred_seq, label_seq in zip(preds, labels):
                pred_labels = []
                gold_labels = []
                for p, l in zip(pred_seq, label_seq):
                    if l != -100:  # ignore padding
                        pred_labels.append(label_list[p])
                        gold_labels.append(label_list[l])
                true_predictions.append(pred_labels)
                true_labels.append(gold_labels)

            # compute seqeval
            results = seqeval.compute(predictions=true_predictions, references=true_labels)

            # store results
            for metric in ["precision", "recall", "f1", "accuracy"]:
                results_dict[model_name][head_name][metric].append(results[metric])

FileNotFoundError: [Errno 2] No such file or directory: '/home/marwas/POLIANNA/notebooks/models/sep/deberta-v3-base_0/pytorch_model.bin'

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"{res}")
        print(results_dict[m][res])

In [ ]:
fn = "2nd_results_separateheads_seqeval"
with open(cwd+f"/outputs/{fn}.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, indent=4)

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"\n{res}")
        df = pd.DataFrame(results_dict[m][res])
        df.loc['mean'] = df.mean()
        print(df)

new metrics -- token micro f1

In [ ]:
mode = "sep"
results_dict = {
    "microsoft/deberta-v3-base":{},
    "FacebookAI/xlm-roberta-base":{},
    "dslim/bert-base-NER-uncased":{}
}
for model_name in list(results_dict):
    results_dict[model_name]["Overall"] = {"precision":[], "recall":[], "f1":[], "accuracy":[]}
    for ftr in ["Actor", "InstrumentType", "Objective", "Resource", "Time"]:
        results_dict[model_name][ftr] = {"precision":[], "recall":[], "f1":[], "number":[]}
for model_name in list(results_dict):
    for r in [0,1,2]:
        dataset_dict = DatasetDict.load_from_disk(cwd+f"/inputs/{mode}/dsdct_r{r}")
        model_tt = AutoModelForTokenClassification.from_pretrained(cwd+f"/models/{mode}/{model_name.split('/')[-1]}_{r}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model_tt.to(device)
        model_tt.eval()
        texts = list(dataset_dict['test']['text'])
        # tokenize batch
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, is_split_into_words=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        # get labels in batch form
        label_cols = ["labels_Actor","labels_InstrumentType","labels_Objective","labels_Resource","labels_Time"]
        labels_batch = {name: torch.stack([torch.tensor(x[name]) for x in dataset_dict['test']]) for name in label_cols}
        with torch.no_grad():
            outputs = model_tt(**inputs, **labels_batch)
            # outputs is TokenClassifierOutput
            logits = outputs.logits  # dict of logits per head
        results = compute_metrics_multihead((logits, labels_batch))
        for k, v in results.items():
            if k[:4] == "over":  # overall metrics
                _, metric = k.split("_")
                results_dict[model_name]["Overall"][metric].append(float(v))
            else:  # per head metrics
                for mtr in v:
                    results_dict[model_name][k][mtr].append(float(v[mtr]))

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"{res}")
        print(results_dict[m][res])

In [ ]:
fn = "2nd_results_separateheads_tokenmicrof1"
with open(cwd+f"/outputs/{fn}.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, indent=4)

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"\n{res}")
        df = pd.DataFrame(results_dict[m][res])
        df.loc['mean'] = df.mean()
        print(df)